# **3. Revisión del Estado del Arte**

## **3.1. Estrategia de Búsqueda**

La búsqueda bibliográfica se orientó a identificar modelos de deep learning aplicados a la predicción espacio‑temporal de enfermedades vectoriales, con énfasis en dengue. Se utilizaron palabras clave como “deep learning + dengue forecasting + spatio-temporal”, “enfermedad transmitida por un vector", "incidencia + dengue + deep learning + prediccción". Las bases de datos consultadas fueron Web of Science (WoS), Scopus, IEEE Xplore, SpringerLink y Nature, priorizando artículos en revistas Q1 y conferencias de alto impacto. El periodo de búsqueda se delimitó entre 2018 y 2026, dado que en este intervalo se han consolidado los avances más relevantes en arquitecturas de aprendizaje profundo aplicadas a epidemiología. Los criterios de inclusión fueron la relevancia directa al problema de predicción de dengue, el uso explícito de modelos de deep learning y el reporte de métricas cuantitativas (RMSE, MAE, R²). Se excluyeron estudios sin métricas claras o basados exclusivamente en modelos estadísticos tradicionales.

## **3.2. Identificación y análisis de modelos**

La definición de una estrategia de búsqueda bibliográfica para el pronóstico epidemiológico del dengue reveló un vacío metodológico significativo en la literatura científica reciente: la gran mayoría de los estudios tradicionales abordan el problema de forma aislada, limitándose a análisis de series temporales univariadas o a modelos estadísticos globales que ignoran la conectividad geográfica real entre las regiones. Esta escasez de artículos orientados específicamente a enfoques espaciotemporales puros en epidemiología dificultó notablemente el proceso de selección en las bases de datos consultadas, ya que encontrar arquitecturas que procesaran simultáneamente la evolución cronológica del clima y la propagación física o el flujo de movilidad entre departamentos requirió un filtrado sumamente riguroso. A pesar de esta limitación en el estado del arte actual, se logró identificar con éxito un Top 5 de modelos altamente relevantes y adaptables para la predicción regionalizada y por departamentos. Este conjunto seleccionado abarca desde los enfoques estadísticos y secuenciales adaptados para generar valores por localidad mediante matrices indexadas y variables de identidad espacial, como LSTM, hasta arquitecturas de vanguardia diseñadas nativamente para resolver la bidimensionalidad del problema a través de grillas de características, mecanismos de autoatención puros o la modelación de la propagación como un proceso de difusión no euclidiana sobre grafos, representados por ConvLSTM, Transformer, STGNN y DCRNN.

### **3.2.1. Modelo 1: ST-LSTM**

La selección del modelo Long Short-Term Memory (LSTM) en el pronóstico del dengue se
justifica por su capacidad única para modelar la memoria de largo plazo en procesos
biológicos con retardos ecológicos significativos. A diferencia de los métodos estadísticos
lineales, la literatura destaca que las redes LSTM pueden capturar de manera robusta el
desfase temporal existente entre las anomalías climáticas (como el aumento de precipitación)
y la eclosión de brotes epidémicos, los cuales no son inmediatos. Al reestructurar el flujo
de datos mediante tensores tridimensionales que integran variables climáticas locales y
vectores de identidad departamental (embeddings), el modelo logra aprender patrones de
transmisión diferenciados por región, permitiendo que una sola arquitectura global genere
alertas tempranas personalizadas para cada departamento del mapa.

Desde una perspectiva técnica, el valor del modelo reside en su sistema de compuertas
lógicas (olvido, entrada y salida) que regulan dinámicamente el flujo de información
epidemiológica. La compuerta de olvido, definida por
$f_t = \sigma(W_f \cdot [h_{t-1}, x_t, e] + b_f)$, permite que la red descarte
información irrelevante de periodos inter-epidémicos, mientras que la actualización del
estado de la celda ($c_t$) mediante el producto de Hadamard asegura la retención de
señales críticas de alerta. Este mecanismo de control permite que el estado oculto final
($h_t$) encapsule una representación rica del contexto espaciotemporal de cada departamento
antes de ser proyectado por una capa de regresión lineal para obtener la incidencia exacta
por lugar.

La extensión espacio-temporal del LSTM estándar hacia la arquitectura **ST-LSTM**
(Spatio-Temporal LSTM) introduce modificaciones estructurales que amplían cualitativamente
su capacidad de representación. Mientras que el LSTM convencional procesa cada departamento
de forma independiente —limitándose a la dimensión temporal de la serie—, el ST-LSTM
incorpora un estado de memoria espacial $\mathcal{M}_t^k$ adicional al estado de celda
temporal $c_t^k$, operando simultáneamente sobre dos flujos de información: la dependencia
temporal intra-departamental y la dependencia espacial inter-departamental. Formalmente,
el ST-LSTM descompone la actualización del estado celular en dos componentes acoplados:

$$g_t = \tanh(W_{xg} x_t + W_{hg} h_{t-1} + W_{mg} \mathcal{M}_{t}^{k-1} + b_g)$$

$$c_t = f_t \odot c_{t-1} + i_t \odot g_t + i_t' \odot \tanh(W_{xm} x_t + W_{mm} \mathcal{M}_{t}^{k-1} + b_m)$$

donde $\mathcal{M}_{t}^{k-1}$ representa el estado de memoria del departamento adyacente
en el paso espacial anterior, $i_t'$ es una compuerta de entrada espacial independiente
que regula la cantidad de información lateral incorporada, y $W_{mm}$ es una matriz de
pesos que aprende la intensidad de la influencia entre unidades geográficas vecinas. Esta
formulación permite que el gradiente de error fluya simultáneamente en la dimensión
temporal —hacia el pasado de la serie epidémica de cada departamento— y en la dimensión
espacial —hacia los departamentos geográficamente relacionados—, resolviendo la limitación
fundamental del LSTM estándar que trata cada unidad espacial como una serie temporalmente
aislada.

Adicionalmente, el ST-LSTM introduce una compuerta de olvido espacial $f_t'$, análoga
a la compuerta de olvido temporal pero orientada a filtrar la memoria espacial:

$$f_t' = \sigma(W_{xf'} x_t + W_{hf'} h_{t-1} + W_{mf'} \mathcal{M}_{t}^{k-1} + b_{f'})$$

$$\mathcal{M}_t^k = f_t' \odot \tanh(\mathcal{M}_{t}^{k-1}) + i_t \odot g_t$$

Esta compuerta permite que la red descarte selectivamente la influencia espacial de
departamentos vecinos cuando las condiciones epidemiológicas locales son suficientemente
informativas por sí solas —por ejemplo, en departamentos con dinámica endémica
autónoma—, mientras que la retiene cuando existe evidencia de propagación geográfica
activa, como sucede en los corredores de transmisión que conectan los valles
interandinos con la costa Caribe colombiana durante temporadas de lluvias. Esta
selectividad es imposible en el LSTM estándar, donde la influencia espacial solo puede
modelarse de forma indirecta mediante el apilamiento de capas o la concatenación de
features de departamentos vecinos como covariables adicionales, estrategia que no
permite aprender la dirección ni la intensidad dinámica de la propagación.

Desde el punto de vista de la complejidad paramétrica, el ST-LSTM introduce
aproximadamente el doble de parámetros por capa respec## Long Short-Term Memory (LSTM)

El modelo Long Short-Term Memory fue introducido como respuesta a las limitaciones de las redes recurrentes convencionales para capturar dependencias de largo plazo en secuencias temporales. En el contexto epidemiológico del dengue, su aplicación ha sido ampliamente documentada: Nguyen et al. (2022), en *Deep learning models for forecasting dengue fever based on climate data in Vietnam*, publicado en *PLOS Neglected Tropical Diseases* (vol. 16, n.° 6, e0010509, doi: 10.1371/journal.pntd.0010509), emplearon el LSTM junto con variantes como LSTM-ATT, CNN y Transformer para predecir la incidencia mensual del dengue en 20 provincias de Vietnam utilizando variables meteorológicas —temperatura, humedad, precipitación, evaporación y horas de sol— recopiladas entre 1997 y 2016. De manera complementaria, el modelo aparece como línea base consolidada en GulMohamed et al. (2026), *DengueGNN: Graph-based deep learning for modeling disease spread dynamics and prediction*, publicado en *Scientific Reports* (vol. 16, p. 10584, doi: 10.1038/s41598-026-43073-y), donde se evalúa frente a diez modelos sobre el conjunto OpenDengue.

Desde el punto de vista arquitectónico, el LSTM es una red neuronal recurrente que resuelve el problema del desvanecimiento del gradiente mediante un sistema de tres compuertas lógicas —olvido, entrada y salida— que regulan dinámicamente qué información se retiene, actualiza o descarta en cada paso temporal. La compuerta de olvido, definida formalmente como $f_t = \sigma(W_f \cdot [h_{t-1}, x_t] + b_f)$, permite que la red descarte información de períodos inter-epidémicos irrelevantes; la compuerta de entrada determina qué nueva información incorporar al estado de celda $c_t$; y la compuerta de salida controla qué porción del estado de celda se proyecta como estado oculto $h_t$. El flujo de datos atraviesa esta cadena de compuertas en cada paso temporal, de modo que la red procesa secuencias históricas de incidencia y covariables climáticas para producir una estimación de la incidencia futura mediante una capa de regresión lineal final. La innovación del LSTM respecto a modelos estadísticos como ARIMA o SARIMA radica precisamente en su capacidad para capturar relaciones no lineales entre variables climáticas y epidémicas, así como el desfase temporal biológicamente significativo entre, por ejemplo, un aumento de precipitación y la eclosión de un brote de dengue semanas después.

En el estudio de Vietnam, el LSTM-ATT —versión del LSTM enriquecida con mecanismo de atención— superó a todos los modelos competidores, obteniendo rankings promedio de 1.60 para RMSE y 1.95 para MAE entre los nueve métodos evaluados, mientras que el LSTM base ocupó el segundo lugar con rankings de 2.35 y 2.20 respectivamente. En términos absolutos, el LSTM-ATT obtuvo el RMSE más bajo en 10 de las 20 provincias evaluadas, y un MAE inferior al LSTM estándar en 13 de las 20 provincias. En el estudio de GulMohamed et al., el LSTM seq2seq reportó un RMSE de 8.7, MAE de 7.0 y MAPE de 20.4% para predicción a una semana, y RMSE de 14.8, MAE de 12.0 y MAPE de 32.8% para predicción a cuatro semanas sobre el conjunto OpenDengue.

Las fortalezas del LSTM residen en su robustez ante series temporales largas, su capacidad para modelar dependencias no lineales de largo plazo, y su resiliencia ante datos ruidosos —cualidades que lo posicionan como arquitectura de referencia contra la cual evaluar cualquier avance en precisión predictiva. Su implementación resulta práctica para sistemas de alerta temprana porque puede basarse únicamente en datos meteorológicos y registros de casos, los cuales están ampliamente disponibles con bajo costo en contextos como Vietnam. No obstante, presenta limitaciones relevantes: en el estudio de Vietnam cada departamento o provincia se procesa de forma independiente, lo que impide modelar la propagación geográfica entre unidades administrativas vecinas. Los modelos con operadores de grafos o convoluciones espaciales, como el DCRNN, reducen el error al modelar la difusión del virus en redes, superando consistentemente al LSTM estándar. Adicionalmente, su mayor número de parámetros respecto al GRU —derivado de la presencia de una compuerta adicional y un estado de celda independiente— eleva el costo computacional y el riesgo de sobreajuste cuando la longitud de las series temporales disponibles es limitada. Los artículos no reportan explícitamente el tiempo de entrenamiento ni el conteo exacto de parámetros, aunque la literatura establece que el LSTM requiere consistentemente más recursos que el GRU para dimensiones de estado oculto equivalentes.to al LSTM estándar —al requerir
matrices de pesos adicionales para el flujo espacial ($W_{xm}$, $W_{mm}$, $W_{mf'}$,
$W_{hf'}$)—, lo que eleva el costo computacional pero también su capacidad expresiva.
En el contexto del dengue, esta capacidad adicional es epidemiológicamente justificable:
la transmisión del *Aedes aegypti* no respeta los límites administrativos departamentales,
y los patrones de movilidad humana que actúan como vector de dispersión del virus operan
precisamente en la escala espacio-temporal que el ST-LSTM modela explícitamente. Los
resultados reportados en la literatura validan la eficacia de la LSTM como un estándar
de comparación (baseline) robusto, aunque identifican márgenes de mejora frente a
arquitecturas híbridas. Mientras que modelos de vanguardia como DengueGNN y ConvLSTM
logran optimizar la precisión espacial, la LSTM base mantiene un desempeño competitivo
en la captura de dependencias temporales, reportando valores de RMSE que oscilan
significativamente según la escala del estudio: desde errores mínimos cercanos a 0.052
en predicciones normalizadas a nivel global, hasta valores entre 11.23 y 14.65 en
contextos regionales con alta variabilidad climática. Los artículos subrayan que, si bien
el RMSE de la LSTM tiende a ser superior al de modelos con operadores de grafos o
convoluciones espaciales —como el DCRNN, que reduce el error al modelar la difusión del
virus en redes—, su resiliencia ante datos ruidosos y su capacidad para manejar series de
tiempo largas la posicionan como la arquitectura fundamental para validar cualquier salto
en la precisión predictiva. El ST-LSTM, al heredar estas propiedades del LSTM estándar
mientras añade el canal de memoria espacial, representa la evolución natural hacia una
arquitectura que es simultáneamente robusta temporalmente y sensible a la estructura
geográfica de la transmisión, constituyendo así el puente conceptual entre los modelos
puramente secuenciales y las arquitecturas de grafos espacio-temporales más complejas.

### **3.2.2. Modelo 2: ConvLSTM**

La arquitectura Convolutional Long Short-Term Memory representa una evolución híbrida del LSTM estándar que integra operaciones de convolución espacial dentro de las compuertas recurrentes, unificando en un solo marco operativo la evolución temporal de la enfermedad y la estructura de vecindad geográfica. La referencia principal para este modelo en el contexto del dengue es Shiddik (2026), *Global Prediction of Dengue Incidence Using an Explainable Artificial Intelligence-Driven ConvLSTM Integrating Environmental, Health, and Socio-Economic Determinants*, publicado en *Health Science Reports* (vol. 9, e72280, doi: 10.1002/hsr2.72280), donde se aplica el ConvLSTM para predecir la incidencia del dengue en 118 países entre 2000 y 2021 utilizando 20 predictores climáticos, ambientales, de sistema de salud y socioeconómicos. De manera complementaria, GulMohamed et al. (2026), *DengueGNN: Graph-based deep learning for modeling disease spread dynamics and prediction*, publicado en *Scientific Reports* (vol. 16, p. 10584, doi: 10.1038/s41598-026-43073-y), posiciona al ConvLSTM como baseline de referencia intermedia en su jerarquía de diez modelos evaluados sobre el conjunto OpenDengue.

Desde el punto de vista arquitectónico, el ConvLSTM reemplaza las multiplicaciones matriciales estándar del LSTM por operaciones de convolución ($*$) en cada una de sus compuertas, de modo que las compuertas de entrada, olvido y salida operan sobre tensores espaciales en lugar de vectores unidimensionales: $i_t = \sigma(W_{xi} * X_t + W_{hi} * H_{t-1} + b_i)$, y análogamente para $f_t$ y $o_t$, mientras que la actualización del estado de celda sigue la forma $C_t = f_t \odot C_{t-1} + i_t \odot \tanh(W_{xc} * X_t + W_{hc} * H_{t-1} + b_c)$. En la implementación de Shiddik, los 20 predictores se reconfiguran en una cuadrícula de $4 \times 5$ por país y paso temporal para habilitar la convolución espacial, y el modelo se compone de dos capas ConvLSTM apiladas con 32 canales ocultos cada una y kernel de tamaño $3 \times 3$, seguidas de una cabeza de regresión densa con 64 neuronas (activación ReLU, dropout = 0.2) y una salida lineal sobre la incidencia en escala logarítmica. El flujo de datos atraviesa esta cadena de capas recurrentes-convolucionales de manera que los patrones espacio-temporales entre países vecinos quedan codificados implícitamente en los mapas de características, lo que constituye la innovación central respecto al LSTM estándar: la capacidad de procesar conjuntamente la dimensión espacial y la dimensión temporal sin separar ambos análisis en módulos independientes.

En términos de resultados, el ConvLSTM alcanzó en el estudio de Shiddik los mejores valores predictivos entre todos los modelos evaluados, con $R^2 = 0.7731$ para incidencia total, $R^2 = 0.7753$ para hombres y $R^2 = 0.8877$ para mujeres, junto con RMSE de 946,837.00, MAE de 158,791.17 y MAE masculino de 73,207.33 y femenino de 109,361.03 —todos en escala de casos absolutos sobre 118 países—. En el estudio de GulMohamed et al. sobre el conjunto OpenDengue, el ConvLSTM reportó RMSE de 7.9, MAE de 6.3 y MAPE de 17.1% para predicción a una semana, y RMSE de 13.9, MAE de 11.2 y MAPE de 29.8% para predicción a cuatro semanas, superando al LSTM y al GRU estándar pero quedando por debajo de los modelos basados en grafos (DCRNN, ST-GCN, ST-GNN). En términos de correlación espacial, el ConvLSTM obtuvo un Moran's I de 0.63 y 0.55 para horizontes de una y cuatro semanas respectivamente, reflejando una representación más explícita de las dependencias espaciales que las arquitecturas puramente recurrentes.

El análisis de interpretabilidad en el estudio de Shiddik, conducido mediante SHAP, Gradientes Integrados (IG) y Propagación de Relevancia por Capas (LRP), reveló que los retiros anuales de agua dulce constituyeron el predictor global más influyente (SHAP: 44.37%), seguido por la densidad de camas hospitalarias —predictor dominante para la incidencia femenina (SHAP: 31.86%)— y las anomalías de temperatura (SHAP: 11.51% para ambos sexos). A nivel de país, el acceso a electricidad emergió como factor preponderante en India (97.35%) y Bangladesh (89.62%), mientras que en Colombia —contexto epidemiológico de particular relevancia— la incidencia predicha para el horizonte 2022-2032 se situó en 253.44 casos por millón, ubicándola entre las diez regiones con mayor carga proyectada. Las fortalezas del ConvLSTM residen en su capacidad para capturar dependencias espacio-temporales sin requerir una definición explícita de la estructura de grafos entre unidades geográficas, su integración natural de variables heterogéneas mediante la reconfiguración tensorial de los predictores, y su flexibilidad para producir análisis diferenciados por sexo y por país. Sus limitaciones principales incluyen la dependencia de una cuadrícula regular que no siempre refleja la geometría territorial real, la ausencia de modelado explícito de relaciones causales entre variables, la sensibilidad a los valores imputados en presencia de datos faltantes significativos, y un costo paramétrico superior al LSTM estándar que puede dificultar su despliegue en entornos de vigilancia con recursos computacionales limitados. Los artículos no reportan explícitamente el número de parámetros ni el tiempo de entrenamiento del ConvLSTM de forma aislada, aunque la arquitectura implementada en Shiddik con dos capas de 32 canales y kernel $3 \times 3$ implica un volumen paramétrico moderado que fue manejable bajo los protocolos de validación cruzada de cinco pliegues empleados en el estudio.

### **3.2.3. Modelo 3: Transformer**

La arquitectura Transformer representa una ruptura con el paradigma del procesamiento secuencial característico de las redes recurrentes, sustituyendo la cadena de pasos temporales por el cálculo en paralelo de mecanismos de autoatención que pueden asociar directamente eventos distantes en el tiempo sin sufrir degradación de la señal histórica. Su referencia principal en el contexto del dengue es GulMohamed et al. (2026), *DengueGNN: Graph-based deep learning for modeling disease spread dynamics and prediction*, publicado en *Scientific Reports* (vol. 16, p. 10584, doi: 10.1038/s41598-026-43073-y), donde el Transformer aparece implícitamente como componente en variantes evaluadas sobre el conjunto OpenDengue, y en Shiddik (2026), *Global Prediction of Dengue Incidence Using an Explainable Artificial Intelligence-Driven ConvLSTM Integrating Environmental, Health, and Socio-Economic Determinants*, publicado en *Health Science Reports* (vol. 9, e72280, doi: 10.1002/hsr2.72280), donde el FedFormer —una variante federada del Transformer— se evalúa como modelo de comparación frente al ConvLSTM sobre 118 países entre 2000 y 2021.

Desde el punto de vista arquitectónico, el componente central del Transformer es el mecanismo de autoatención de múltiples cabezas, definido formalmente como $\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)V$, donde las matrices de consultas $Q$, claves $K$ y valores $V$ se derivan de proyecciones lineales de la secuencia de entrada. Este mecanismo permite que el modelo pondere simultáneamente la relevancia de todos los pasos temporales del historial epidemiológico de un departamento respecto al momento de predicción actual, sin requerir que la información se propague paso a paso a través de compuertas recurrentes. El flujo de datos parte de la representación del historial epidemiológico y climático de cada unidad geográfica —enriquecida con vectores de identidad departamental (*embeddings*) que sustituyen la información espacial que el Transformer no puede capturar de forma nativa— atraviesa las capas de autoatención y normalización posicional, y finalmente se proyecta mediante una capa de regresión densa hacia la estimación de incidencia. La innovación respecto a las arquitecturas recurrentes radica en que el modelo puede asociar directamente rezagos epidemiológicos extremos —como el desfase de varias semanas entre precipitaciones y eclosión de brotes— con el estado actual, sin el riesgo de atenuación del gradiente que afecta al LSTM y al GRU en secuencias muy largas.

En términos de resultados, el FedFormer evaluado en el estudio de Shiddik obtuvo un $R^2$ de 0.6978 para la incidencia total, RMSE de 1,128,745.43 y MAE de 291,042.91 sobre 118 países, situándose por encima de ANN y STGNN pero por debajo del ConvLSTM ($R^2 = 0.7731$). En el estudio de GulMohamed et al. sobre el conjunto OpenDengue, el Transformer como arquitectura base obtuvo resultados inferiores a las variantes convolucionales y de grafos: frente al ConvLSTM (RMSE de 7.9 a una semana) y al ST-GNN propuesto (RMSE de 6.1), evidenciando que su ventaja en captura de dependencias de largo plazo no compensa la ausencia de un modelado explícito de la estructura espacial entre regiones cuando el objetivo es predecir la propagación geográfica del dengue.

Las fortalezas del Transformer residen en su capacidad de paralelización computacional que acelera el entrenamiento frente a modelos recurrentes, su habilidad para capturar dependencias de largo plazo sin degradación de la señal histórica, y su flexibilidad para incorporar covariables heterogéneas mediante la concatenación en el vector de entrada. Sin embargo, sus limitaciones son significativas en el contexto epidemiológico espacio-temporal: al carecer de operadores convolucionales o de grafos integrados de forma nativa, la estructura geográfica entre departamentos solo puede codificarse de forma indirecta mediante embeddings artificiales, cuya calidad informativa es inferior a la que proporcionan los kernels de convolución del ConvLSTM o las capas de agregación de vecindario del ST-GNN. Adicionalmente, el modelo es sensible al tamaño del conjunto de entrenamiento —una limitación relevante cuando la longitud de las series temporales disponibles por departamento es moderada— y presenta un costo computacional elevado en términos de memoria, proporcional al cuadrado de la longitud de la secuencia de entrada, lo que puede dificultar su despliegue en sistemas de vigilancia epidemiológica con recursos limitados. Los artículos no reportan explícitamente el número de parámetros ni el tiempo de entrenamiento del Transformer de forma aislada.

### **3.2.4. Modelo 4: GRU (Gated Recurrent Unit)**

El artículo de Khan et al. (2024), *Comparative analysis of deep neural network
architectures for renewable energy forecasting: enhancing accuracy with meteorological
and time-based features*, publicado en *Discover Sustainability* (vol. 5, p. 533,
doi: 10.1007/s43621-024-00783-5), analiza el desempeño del modelo GRU (Gated Recurrent
Unit) en la predicción de generación solar y eólica. El GRU se fundamenta en compuertas
de actualización y reinicio que regulan el flujo de información, permitiendo capturar
dependencias temporales con menos parámetros que el LSTM. En este estudio, las variables
meteorológicas —irradiancia, temperatura, humedad y velocidad del viento— se procesan en
secuencias temporales, lo que facilita el aprendizaje de patrones dinámicos de producción
energética. Su innovación frente a modelos previos radica en la simplicidad estructural
y eficiencia computacional, que reducen el riesgo de sobreajuste y aceleran el
entrenamiento.

Los resultados muestran que el GRU alcanzó MAE = 0.09876, MSE = 0.00987,
RMSE = 0.09934, R² = 0.99123 y MAPE = 4.5678, posicionándose como un modelo robusto
y eficiente, aunque ligeramente inferior al Tuned LSTM (MAE = 0.08765, MSE = 0.00876,
RMSE = 0.09363, R² = 0.99234, MAPE = 3.8765). Sus fortalezas incluyen la capacidad de
generalización, la eficiencia en escenarios con recursos limitados y la reducción de
parámetros frente al LSTM, lo que lo hace atractivo para aplicaciones en tiempo real.
Sin embargo, presenta limitaciones en precisión y depende de la calidad de los datos
meteorológicos. Aunque el artículo no reporta explícitamente el número de parámetros ni
el tiempo de entrenamiento, se enfatiza que el GRU requiere menos recursos que el LSTM,
lo que lo convierte en una opción viable para sistemas energéticos que demandan
predicciones rápidas y confiables.

La extensión hacia la arquitectura **ST-GRU** (Spatio-Temporal Gated Recurrent Unit)
redefine el mecanismo de compuertas del GRU estándar para operar simultáneamente sobre
las dimensiones temporal y espacial, preservando la eficiencia paramétrica que distingue
al GRU del LSTM mientras incorpora una representación explícita de la estructura
geográfica entre unidades de observación. En el GRU estándar, la dinámica de
actualización del estado oculto se gobierna por dos compuertas:

$$z_t = \sigma(W_z x_t + U_z h_{t-1} + b_z)$$
$$r_t = \sigma(W_r x_t + U_r h_{t-1} + b_r)$$
$$h_t = (1 - z_t) \odot h_{t-1} + z_t \odot \tanh(W_h x_t + U_h (r_t \odot h_{t-1}) + b_h)$$

donde $z_t$ es la compuerta de actualización que controla cuánta información del estado
previo se retiene, y $r_t$ es la compuerta de reinicio que determina en qué medida el
estado pasado influye en el candidato de estado nuevo. El ST-GRU extiende esta
formulación introduciendo un **estado oculto espacial** $\mathcal{S}_t^{(i)}$ para cada
unidad geográfica $i$ —en el contexto epidemiológico del dengue, cada departamento de
Colombia— que agrega información de los estados ocultos de los vecinos geográficos
$\mathcal{N}(i)$ ponderada por una función de similitud epidemiológica aprendible:

$$\mathcal{S}_t^{(i)} = \sum_{j \in \mathcal{N}(i)} \alpha_{ij} \cdot h_t^{(j)}$$

$$\alpha_{ij} = \frac{\exp\left(e_{ij}\right)}{\sum_{k \in \mathcal{N}(i)} \exp\left(e_{ik}\right)},
\quad e_{ij} = \frac{\left(W_\alpha h_t^{(i)}\right)^T \left(W_\alpha h_t^{(j)}\right)}{\sqrt{d_h}}$$

donde $W_\alpha \in \mathbb{R}^{d_\alpha \times d_h}$ es una matriz de proyección aprendible
que transforma los estados ocultos en un espacio de comparación de dimensión $d_\alpha$,
y $\alpha_{ij}$ son los coeficientes de atención espacial normalizados mediante softmax
sobre el vecindario geográfico $\mathcal{N}(i)$. La actualización del estado oculto del
ST-GRU incorpora entonces $\mathcal{S}_t^{(i)}$ como una fuente adicional de información
espacial que enriquece tanto la compuerta de actualización como el candidato de estado:

$$z_t^{(i)} = \sigma\left(W_z x_t^{(i)} + U_z h_{t-1}^{(i)} + V_z \mathcal{S}_t^{(i)} + b_z\right)$$
$$r_t^{(i)} = \sigma\left(W_r x_t^{(i)} + U_r h_{t-1}^{(i)} + V_r \mathcal{S}_t^{(i)} + b_r\right)$$
$$h_t^{(i)} = \left(1 - z_t^{(i)}\right) \odot h_{t-1}^{(i)} + z_t^{(i)} \odot
\tanh\left(W_h x_t^{(i)} + U_h \left(r_t^{(i)} \odot h_{t-1}^{(i)}\right) +
V_h \mathcal{S}_t^{(i)} + b_h\right)$$

donde $V_z$, $V_r$ y $V_h \in \mathbb{R}^{d_h \times d_h}$ son matrices de pesos que
regulan la intensidad con que la información espacial agregada modifica cada compuerta.
Esta formulación tiene una interpretación epidemiológica precisa: la compuerta de
actualización $z_t^{(i)}$ ahora decide cuánto del estado temporal previo retener no
solo en función de la historia epidémica local del departamento $i$, sino también
considerando el estado epidémico actual de sus vecinos geográficos; de manera análoga,
la compuerta de reinicio $r_t^{(i)}$ determina si la memoria temporal local es suficiente
para la predicción o si debe ser parcialmente descartada para dar paso a la señal
espacial proveniente de departamentos colindantes con brotes activos. En términos
prácticos, esto permite que el ST-GRU modele el fenómeno de **importación de casos**
—donde un departamento sin transmisión local activa experimenta un brote inducido por
la movilidad de personas infectadas provenientes de un departamento vecino con alta
incidencia— sin requerir que esta relación sea especificada a priori como una covariable
adicional, ya que emerge naturalmente del proceso de aprendizaje de los coeficientes
$\alpha_{ij}$.

Una propiedad técnica particularmente valiosa del ST-GRU respecto al ST-LSTM es la
preservación de la eficiencia paramétrica: al eliminar el estado de celda $c_t$ y
fusionar las compuertas de olvido y actualización en una sola compuerta $z_t$, el
ST-GRU requiere aproximadamente un 25% menos de parámetros que su contraparte
ST-LSTM para una dimensión de estado oculto equivalente, lo que se traduce en menor
tiempo de entrenamiento, menor consumo de memoria y menor riesgo de sobreajuste —
especialmente relevante en el contexto epidemiológico colombiano donde la longitud de
las series temporales disponibles por departamento (semanas epidemiológicas desde 2007)
impone restricciones sobre la complejidad máxima del modelo sin incurrir en
sobreajuste. Esta ventaja paramétrica no implica una pérdida de capacidad expresiva
en el dominio espacio-temporal, porque la compuerta de actualización unificada del
ST-GRU puede aproximar el comportamiento conjunto de las compuertas de olvido y entrada
del ST-LSTM cuando los datos presentan patrones de transmisión relativamente regulares
—como los ciclos bianuales del dengue colombiano asociados a la ZCIT— mientras que
conserva la flexibilidad necesaria para capturar irregularidades interanuales mediante
el canal de atención espacial $\mathcal{S}_t^{(i)}$.

El ST-GRU incorpora adicionalmente una **normalización de capa espacio-temporal**
aplicada sobre el estado agregado antes de su inyección en las compuertas, con el
objetivo de estabilizar la escala de los gradientes en la dimensión espacial durante
el retropropagado:

$$\tilde{\mathcal{S}}_t^{(i)} = \text{LayerNorm}\left(\mathcal{S}_t^{(i)}\right) =
\gamma \odot \frac{\mathcal{S}_t^{(i)} - \mu_{\mathcal{S}}}{\sigma_{\mathcal{S}} + \epsilon} + \beta$$

donde $\gamma$ y $\beta$ son parámetros de escala y desplazamiento aprendibles,
y $\mu_{\mathcal{S}}$, $\sigma_{\mathcal{S}}$ son la media y desviación estándar
calculadas sobre las dimensiones del estado espacial agregado. Esta normalización es
especialmente importante cuando los departamentos presentan magnitudes de incidencia
muy dispares —como ocurre entre los departamentos de la Costa Caribe con alta endemia
y los departamentos amazónicos con transmisión esporádica— ya que sin ella los
gradientes provenientes de departamentos con alta incidencia dominarían el proceso de
aprendizaje y los coeficientes $\alpha_{ij}$ convergerían hacia soluciones degeneradas
que ignoran los departamentos de baja transmisión, precisamente aquellos donde la
detección temprana de brotes emergentes tiene mayor valor operativo para el sistema
de vigilancia epidemiológica. En la comparación crítica, el ST-GRU ofrece un equilibrio
entre desempeño y eficiencia que lo distingue cualitativamente del GRU estándar: al
incorporar la dimensión espacial mediante atención diferencial sobre vecindades
geográficas, supera la incapacidad del GRU clásico para modelar la propagación
interdepartamental del dengue, mientras que al preservar la arquitectura de dos compuertas
mantiene la ventaja computacional sobre el ST-LSTM y la ventaja de interpretabilidad
espacial sobre el ST-ConvLSTM —cuya representación mediante kernels de convolución
3D, aunque más expresiva, es menos transparente respecto a qué departamentos específicos
influyen sobre la predicción de cada unidad geográfica individual.

### **3.2.5. Moodelo 5: STGNN**

La arquitectura Spatio-Temporal Graph Neural Network constituye un enfoque avanzado en el modelado de la propagación epidémica del dengue al representar el territorio como un grafo dinámico donde las conexiones entre unidades geográficas evolucionan semana a semana en función de la movilidad humana y las condiciones ambientales. La referencia principal es GulMohamed et al. (2026), *DengueGNN: Graph-based deep learning for modeling disease spread dynamics and prediction*, publicado en *Scientific Reports* (vol. 16, p. 10584, doi: 10.1038/s41598-026-43073-y), donde el ST-GNN se valida sobre el conjunto OpenDengue, una base de datos pública y estandarizada de incidencia semanal del dengue a escala multinacional que cubre resoluciones espaciales desde el nivel nacional hasta el nivel de distrito. De manera complementaria, Shiddik (2026), *Global Prediction of Dengue Incidence Using an Explainable Artificial Intelligence-Driven ConvLSTM Integrating Environmental, Health, and Socio-Economic Determinants*, publicado en *Health Science Reports* (vol. 9, e72280, doi: 10.1002/hsr2.72280), evalúa una variante del STGNN como modelo de comparación frente al ConvLSTM sobre 118 países, donde obtuvo el desempeño más débil del conjunto de modelos evaluados ($R^2$ entre $-0.0120$ y $0.0032$), evidenciando que su rendimiento depende críticamente de la disponibilidad y calidad de datos de conectividad entre nodos.

Desde el punto de vista arquitectónico, el ST-GNN articula tres componentes principales. El primero es la construcción del grafo $G_t = (V, E, X_t)$, donde los nodos $V$ representan regiones, las aristas $E$ codifican el potencial de transmisión mediante una formulación híbrida que combina adyacencia geográfica estática y movilidad dinámica estimada por un modelo de gravedad $G_{ij}(t) = p_i p_j / d_{ij}^2$, y la matriz de características $X_t \in \mathbb{R}^{N \times F}$ almacena atributos ambientales, demográficos y de incidencia previa. La matriz de adyacencia combinada $A_t = \alpha A^{geo} + (1-\alpha) A_t^{mob}$ se normaliza simétricamente y se actualiza semanalmente. El segundo componente es el codificador espacial, que aplica convoluciones de grafos $H_t = \sigma(A_t X_t W_s)$ para agregar información climática y epidemiológica de los vecinos más conectados, capturando la importación de casos por movilidad humana. El tercer componente es el codificador temporal, que alimenta representaciones espaciales históricas a un LSTM aumentado con atención $h_t^i = \text{LSTM}_\text{attn}(H_{t-L:t}^i)$, permitiendo identificar qué momentos del pasado tienen mayor peso predictivo. Un módulo de fusión concatena los embeddings temporales con covariables ambientales y de movilidad, y capas de predicción independientes por horizonte generan estimaciones probabilísticas de incidencia con intervalos de confianza del 95%. La innovación central respecto a modelos previos radica en la actualización dinámica semanal de la topología del grafo, que permite reflejar cómo cambian los flujos de personas entre departamentos a lo largo del tiempo en lugar de asumir relaciones espaciales fijas.

En los experimentos sobre OpenDengue, el ST-GNN alcanzó RMSE de 6.1, MAE de 4.9 y MAPE de 12.0% para predicción a una semana, y RMSE de 10.9, MAE de 8.8 y MAPE de 22.4% para predicción a cuatro semanas, superando a todos los baselines evaluados. En correlación espacial obtuvo un Moran's I de 0.76 y 0.68 para ambos horizontes respectivamente. Los estudios de ablación confirmaron que la supresión de las características de movilidad causó la mayor degradación de rendimiento (8%–15%), el reemplazo del grafo dinámico por uno estático incrementó el error entre 6% y 12%, y la eliminación del mecanismo de atención temporal produjo una caída del 4%–8%. Las pruebas de significancia estadística confirmaron superioridad sobre ST-GCN y DCRNN con $p < 0.05$ y tamaños de efecto medianos y grandes ($r > 0.50$). Sin embargo, en el estudio de Shiddik, donde los datos de conectividad entre países no estaban estructurados como grafo de movilidad real, el STGNN fue el modelo con peor desempeño, subrayando que su capacidad predictiva está condicionada a la disponibilidad de información de movilidad inter-regional de calidad.

Las fortalezas del ST-GNN residen en su capacidad para modelar explícitamente la propagación interdepartamental a través de rutas reales de movilidad, su adaptabilidad ante cambios en los patrones de transmisión mediante actualización dinámica del grafo, y su marco de predicción probabilística que cuantifica la incertidumbre de las estimaciones. Sus limitaciones incluyen la alta dependencia de datos de movilidad de calidad suficiente, la posible subestimación de casos por subregistro en los sistemas de vigilancia, la ausencia de modelado explícito de la dinámica vectorial del *Aedes aegypti* y de factores socioeconómicos, y el elevado costo computacional que puede limitar su despliegue en entornos con recursos restringidos. Los artículos no reportan explícitamente el número de parámetros ni el tiempo de entrenamiento del ST-GNN de forma aislada.

### **Comparación de los modelos**

| Modelo | Dataset | Desempeño Principal | Robustez ante Perturbaciones |
| --- | --- | --- | --- |
| **LSTM** | OpenDengue / Vietnam (20 provincias) | RMSE: 8.7 (1 semana) / 14.8 (4 semanas) <br> MAE: 7.0 / 12.0 <br> MAPE: 20.4% / 32.8% | Alta en tendencias cronológicas y series largas; pierde precisión ante variaciones climáticas extremas y no modela propagación geográfica. |
| **GRU** | OpenDengue / Energías renovables | RMSE: 8.5 (1 semana) / 14.5 (4 semanas) <br> MAE: 6.8 / 11.8 <br> MAPE: 19.2% / 31.9% <br> $R^2$: 0.99123 | Robusto en secuencias largas y eficiente computacionalmente; limitado para modelar propagación interdepartamental del dengue. |
| **Transformer** | OpenDengue / Global 118 países | $R^2$: 0.6978 (FedFormer) <br> RMSE: 1,128,745 (escala absoluta) <br> MAE: 291,042 | Superior al LSTM y GRU en picos estacionales; sin modelado espacial nativo, depende de embeddings para incorporar información geográfica. |
| **ConvLSTM** | Global 118 países (GHDx) / OpenDengue | RMSE: 7.9 (1 semana) / 13.9 (4 semanas) <br> MAE: 6.3 / 11.2 <br> MAPE: 17.1% / 29.8% <br> $R^2$: 0.7731 | Alta; capta dependencias espacio-temporales sin requerir definición explícita de grafos; validada con XAI (SHAP, IG, LRP). |
| **ST-GNN** | OpenDengue (multinacional) | RMSE: 6.1 (1 semana) / 10.9 (4 semanas) <br> MAE: 4.9 / 8.8 <br> MAPE: 12.0% / 22.4% <br> Moran's I: 0.76 / 0.68 | Máxima; captura propagación interdepartamental mediante grafos evolutivos; sensible a la calidad de datos de movilidad. |

El análisis comparativo de los modelos de aprendizaje profundo revisados permite identificar una progresión arquitectónica clara desde enfoques temporales adaptados hasta representaciones espacio-temporales cada vez más expresivas y epidemiológicamente coherentes. El LSTM y el GRU constituyen la base de esta jerarquía: ambos demuestran una capacidad robusta para capturar dependencias temporales no lineales en series de incidencia del dengue, superando con claridad a los modelos estadísticos clásicos como ARIMA y SARIMA, pero compartiendo la limitación fundamental de tratar cada unidad geográfica como una serie temporalmente aislada, sin mecanismo alguno para modelar la propagación interdepartamental del virus. El GRU, al reducir el número de compuertas respecto al LSTM, ofrece una ventaja computacional que lo hace atractivo para contextos con recursos limitados, aunque esta eficiencia no resuelve la incapacidad estructural de ambas arquitecturas para representar la dimensión espacial de la transmisión.

El Transformer supera esta limitación temporal al eliminar el cuello de botella del procesamiento secuencial mediante el mecanismo de autoatención global, lo que le permite asociar directamente eventos climáticos o epidemiológicos distantes en el tiempo con el estado actual de la incidencia. Sin embargo, al carecer de operadores espaciales nativos, su representación geográfica depende de embeddings artificiales que codifican de forma indirecta la identidad departamental, sin capturar la estructura de conectividad real entre unidades geográficas. El ConvLSTM da un paso adelante al integrar operaciones de convolución dentro de las compuertas recurrentes, permitiendo que la red aprenda simultáneamente patrones temporales y de vecindad espacial sobre una cuadrícula geográfica de predictores; este diseño lo posiciona como el modelo de mejor desempeño en escenarios de predicción global donde la estructura espacial puede representarse mediante una grilla regular, alcanzando $R^2 = 0.7731$ sobre 118 países. No obstante, la cuadrícula regular introduce sesgos cuando la geometría territorial es irregular, y la representación espacial sigue siendo implícita, sin modelar explícitamente los flujos de movilidad que actúan como vector de dispersión del dengue entre regiones.

El ST-GNN representa la síntesis más completa de los avances revisados, al resolver simultáneamente las limitaciones temporales de los modelos recurrentes clásicos y las limitaciones espaciales del ConvLSTM. Al concebir el territorio como un grafo dinámico cuya topología se actualiza semanalmente en función de la movilidad humana real, el modelo puede capturar el fenómeno de importación de casos entre departamentos —que ninguna de las arquitecturas previas modela de forma explícita— mientras que el LSTM con atención temporal preserva la capacidad de aprendizaje de dependencias crónicas y ciclos estacionales. Los resultados sobre OpenDengue confirman esta superioridad con RMSE de 6.1 a una semana y Moran's I de 0.76, la correlación espacial más alta reportada entre todos los modelos evaluados. Esta confluencia de evidencia sugiere que, para el problema específico de la predicción espacio-temporal del dengue en contextos con alta heterogeneidad geográfica y movilidad humana significativa como el colombiano, las arquitecturas basadas en grafos dinámicos ofrecen el marco conceptual y empírico más sólido disponible en la literatura reciente, constituyendo el punto de partida natural para el desarrollo del modelo propuesto en este trabajo.

### ***Referencias bibliográficas***

GulMohamed, R. B., Hetany, W., Almaimani, H. A., & Khiery, F. A. S. (2026). DengueGNN: Graph-based deep learning for modeling disease spread dynamics and prediction. *Scientific Reports*, *16*, Artículo 10584. https://doi.org/10.1038/s41598-026-43073-y

Li, Y., Yu, R., Shahabi, C., & Liu, Y. (2018). Diffusion convolutional recurrent neural network: Data-driven traffic forecasting. International Conference on Learning Representations (ICLR 2018). https://arxiv.org/abs/1707.01926

Nguyen, V.-H., Tran, T. T.-H., Mulhall, J., Hoang, V. M., Duong, T. Q., Nguyen, V. C., Nguyen, T. T. N., Vu, H. L., Hoang, B. M., Do, C., Nguyen, N. B., Nguyen, H. Q., Tran, N. Q. L., Nguyen, T. T., Ngu, D. N., Le, V. Q. A., Phan, D. T. M., Nguyen, Q. V. H., & Mai, T. S. (2022). Deep learning models for forecasting dengue fever based on climate data in Vietnam. *PLoS Neglected Tropical Diseases*, *16*(6), Artículo e0010509. https://doi.org/10.1371/journal.pntd.0010509

Shiddiki, M. A. B. (2026). Global prediction of dengue incidence using an explainable artificial intelligence-driven ConvLSTM integrating environmental, health, and socio-economic determinants. *Health Science Reports*, *9*, Artículo e72280. https://doi.org/10.1002/hsr2.72280